In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.stats import hypergeom, spearmanr

from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

def show_table(x):
    print(x.to_string() if hasattr(x, 'to_string') else x)


pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


DATA_PATH = "Global_Pollution_Analysis.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS = 5
N_BINS = 3
MIN_SUPPORT = 0.05
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.20
MAX_LEN = 3
N_SHUFFLES = 30
ALPHA = 0.05
LEVELS = ["Low", "Medium", "High"]

if not os.path.exists('/content/Global_Pollution_Analysis.csv') and not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception:
        pass

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError('Global_Pollution_Analysis.csv not found. Upload it next to this notebook.')

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print("Shape:", df.shape)
show_table(df.head())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

def find_col(df, *keyword_sets):
    for kws in keyword_sets:
        if isinstance(kws, str):
            kws = (kws,)
        for c in df.columns:
            if all(k in c.lower() for k in kws):
                return c
    return None

COL = {
    "country":    find_col(df, "country"),
    "year":       find_col(df, "year"),
    "air":        find_col(df, ("air", "pollution"), "air"),
    "water":      find_col(df, ("water", "pollution"), "water"),
    "soil":       find_col(df, ("soil", "pollution"), "soil"),
    "co2":        find_col(df, "co2"),
    "waste":      find_col(df, ("industrial", "waste")),
    "energy_pc":  find_col(df, ("energy", "capita")),
    "energy_tot": find_col(df, ("energy", "consum")),
    "recovery":   find_col(df, "recover"),
    "population": find_col(df, "population"),
}
print("\nDetected columns:")
for k, v in COL.items():
    print(f"  {k:<10} -> {v}")
required = ["country", "year", "air", "water", "soil", "recovery"]
missing_required = [k for k in required if COL[k] is None]
if missing_required:
    raise ValueError(f"Could not find required columns {missing_required}. Available: {list(df.columns)}")


if COL["energy_pc"] is not None:
    df["Energy_Per_Capita"] = df[COL["energy_pc"]]
elif COL["energy_tot"] is not None and COL["population"] is not None:
    df["Energy_Per_Capita"] = df[COL["energy_tot"]] / df[COL["population"]].replace(0, np.nan)
    print("Energy_Per_Capita computed as energy consumption / population")
else:
    raise ValueError("No energy-per-capita (or energy + population) columns found.")


df["Country_Encoded"] = LabelEncoder().fit_transform(df[COL["country"]].astype(str))
df["Year_Encoded"] = LabelEncoder().fit_transform(df[COL["year"]].astype(str))


INDEX_COLS = [COL["air"], COL["water"], COL["soil"]]
filled = df[INDEX_COLS].fillna(df[INDEX_COLS].median())
for c, v in zip(INDEX_COLS, MinMaxScaler().fit_transform(filled).T):
    df[c + "_Scaled"] = v
df["Pollution_Score"] = ((filled - filled.mean()) / filled.std(ddof=0)).mean(axis=1)

show_table(df[[COL["country"], COL["year"]] + INDEX_COLS + ["Energy_Per_Capita", COL["recovery"], "Pollution_Score"]].head())


yearly = df.groupby(COL["year"]).agg(records=("Pollution_Score", "size"),
                                     pollution=("Pollution_Score", "mean"),
                                     recovery=(COL["recovery"], "mean")).reset_index()
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(yearly[COL["year"]], yearly["pollution"], marker="o", color="tomato")
ax1.set_xlabel("Year"); ax1.set_ylabel("Avg pollution score (z)", color="tomato")
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(yearly[COL["year"]], yearly["recovery"], marker="s", color="teal")
ax2.set_ylabel("Avg energy recovered", color="teal")
plt.title("Yearly averages: pollution vs. energy recovery"); plt.tight_layout(); plt.show()

rho_y, p_y = spearmanr(yearly["pollution"], yearly["recovery"])
rho_t, p_t = spearmanr(df[COL["year"]], df["Pollution_Score"])
print(f"Records per year: {yearly['records'].min()}-{yearly['records'].max()} (small groups -> noisy averages)")
print(f"Yearly avg pollution vs yearly avg energy recovery: rho = {rho_y:.2f}, p = {p_y:.3f}")
print(f"Pollution score vs year (is pollution trending?):   rho = {rho_t:.2f}, p = {p_t:.3f}")

BASKET_VARS = {k: v for k, v in {
    "Air": COL["air"], "Water": COL["water"], "Soil": COL["soil"],
    "CO2": COL["co2"], "Waste": COL["waste"],
    "EnergyPC": "Energy_Per_Capita", "Recovery": COL["recovery"]}.items() if v is not None}
POLLUTION_PREFIXES = ("Air", "Water", "Soil", "CO2", "Waste")
print("Basket variables:", BASKET_VARS)

def fit_bins(train_df):
    edges = {n: np.nanquantile(train_df[c], np.linspace(0, 1, N_BINS + 1)[1:-1]) for n, c in BASKET_VARS.items()}
    medians = {n: train_df[c].median() for n, c in BASKET_VARS.items()}
    return edges, medians

def make_basket(part_df, edges, medians):
    cols = {}
    for n, c in BASKET_VARS.items():
        level = np.digitize(part_df[c].fillna(medians[n]).values, edges[n])
        for i, lab in enumerate(LEVELS):
            cols[f"{n}={lab}"] = (level == i)
    return pd.DataFrame(cols, index=part_df.index)

edges_all, med_all = fit_bins(df)
basket_all = make_basket(df, edges_all, med_all)
print("Thresholds (Low|Medium|High cut-offs):")
show_table(pd.DataFrame(edges_all, index=["33rd pct", "67th pct"]).T.round(2))
print("Basket shape:", basket_all.shape)
show_table(basket_all.head(3))

def _assoc(fi, n_rows, **kw):
    try:
        return association_rules(fi, num_itemsets=n_rows, **kw)
    except TypeError:
        return association_rules(fi, **kw)

def bh_qvalues(p):
    p = np.asarray(p, dtype=float); m = len(p)
    order = np.argsort(p); ranked = p[order] * m / (np.arange(m) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m); out[order] = np.minimum(q, 1.0)
    return out

def rule_name(ant, con):
    return " & ".join(sorted(ant)) + "  ->  " + " & ".join(sorted(con))

def mine_rules(basket, min_support=MIN_SUPPORT, min_conf=MIN_CONFIDENCE, max_len=MAX_LEN):
    N = len(basket)
    fi = apriori(basket, min_support=min_support, use_colnames=True, max_len=max_len)
    if fi.empty:
        return fi, pd.DataFrame()
    rules = _assoc(fi, N, metric="confidence", min_threshold=min_conf)
    rules = rules[rules["lift"] > 1.0].copy()
    if rules.empty:
        return fi, rules
    rules["n_A"] = np.rint(rules["antecedent support"] * N).astype(int)
    rules["n_C"] = np.rint(rules["consequent support"] * N).astype(int)
    rules["n_AC"] = np.rint(rules["support"] * N).astype(int)
    rules["p_value"] = hypergeom.sf(rules["n_AC"] - 1, N, rules["n_C"], rules["n_A"])
    rules["q_value"] = bh_qvalues(rules["p_value"].values)
    rules["rule"] = [rule_name(a, c) for a, c in zip(rules["antecedents"], rules["consequents"])]
    return fi, rules.sort_values("lift", ascending=False).reset_index(drop=True)

freq_itemsets, rules = mine_rules(basket_all)
rules["strong"] = (rules["lift"] >= MIN_LIFT)
print(f"Frequent itemsets: {len(freq_itemsets)}  |  rules (confidence >= {MIN_CONFIDENCE}, lift > 1): {len(rules)}")
print(f"Strong rules (lift >= {MIN_LIFT}): {int(rules['strong'].sum())}")
print(f"Rules significant after FDR correction (q < {ALPHA}): {int((rules['q_value'] < ALPHA).sum())}")
print("\nItemset sizes:", freq_itemsets["itemsets"].apply(len).value_counts().sort_index().to_dict())
show = ["rule", "support", "confidence", "lift", "n_AC", "p_value", "q_value"]
print("\nTop 10 rules by lift:")
show_table(rules[show].head(10).round(3))

def is_pollution_to_recovery(row):
    ant, con = row["antecedents"], row["consequents"]
    return (all(a.startswith(POLLUTION_PREFIXES) for a in ant)
            and len(con) == 1 and next(iter(con)).startswith("Recovery="))

pr_rules = rules[rules.apply(is_pollution_to_recovery, axis=1)] if "Recovery" in BASKET_VARS else rules.iloc[0:0]
print(f"Pollution -> energy-recovery rules: {len(pr_rules)}")
show_table(pr_rules[show].head(12).round(3))
print("\nThose with 'High' pollution in the antecedent:")
hi_pr = pr_rules[pr_rules["antecedents"].apply(lambda a: any(x.endswith("=High") for x in a))]
show_table(hi_pr[show].head(8).round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))


fi2 = freq_itemsets[freq_itemsets["itemsets"].apply(len) >= 2].sort_values("support", ascending=False).head(12)
labels = [" & ".join(sorted(s)) for s in fi2["itemsets"]]
axes[0].barh(labels[::-1], fi2["support"].values[::-1], color="teal")
axes[0].axvline(MIN_SUPPORT, color="red", linestyle="--", label=f"min support = {MIN_SUPPORT}")
axes[0].set_title("Most frequent itemsets (2+ items)"); axes[0].set_xlabel("Support"); axes[0].legend()


sc = axes[1].scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="YlOrRd", s=40, edgecolors="k", linewidths=0.3)
plt.colorbar(sc, ax=axes[1], label="Lift")
axes[1].set_title("All rules: support vs confidence"); axes[1].set_xlabel("Support"); axes[1].set_ylabel("Confidence")
plt.tight_layout(); plt.show()


single = rules[(rules["antecedents"].apply(len) == 1) & (rules["consequents"].apply(len) == 1)].copy()
single["A"] = single["antecedents"].apply(lambda s: next(iter(s)))
single["C"] = single["consequents"].apply(lambda s: next(iter(s)))
heat = single.pivot_table(index="A", columns="C", values="lift")

fig, axes = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={"width_ratios": [1.1, 1]})
sns.heatmap(heat, cmap="YlOrRd", center=1, ax=axes[0], cbar_kws={"label": "Lift"}, linewidths=0.3, linecolor="white")
axes[0].set_title("Lift between single items (A -> C)"); axes[0].set_xlabel("Consequent"); axes[0].set_ylabel("Antecedent")

top = rules.head(12)
G = nx.DiGraph()
for _, r in top.iterrows():
    for a in r["antecedents"]:
        for c in r["consequents"]:
            G.add_edge(a, c, lift=r["lift"])
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=1.2)
palette = {"Air": "tomato", "Water": "steelblue", "Soil": "sienna", "CO2": "gray", "Waste": "mediumpurple",
           "EnergyPC": "sandybrown", "Recovery": "teal"}
colors = [palette.get(n.split("=")[0], "darkgray") for n in G.nodes]
nx.draw_networkx(G, pos, ax=axes[1], node_color=colors, node_size=1500, font_size=8, arrows=True,
                 width=[1 + 2 * (G[u][v]["lift"] - 1) for u, v in G.edges], edge_color="dimgray", alpha=0.9)
axes[1].set_title("Network of the 12 highest-lift rules (arrow: antecedent -> consequent)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

def shuffled_copy(frame, seed):
    rng = np.random.RandomState(seed)
    out = frame.copy()
    for c in BASKET_VARS.values():
        out[c] = rng.permutation(out[c].values)
    return out

def evaluate_rules_on(basket, rules_df):
    B = basket.values; idx = {c: i for i, c in enumerate(basket.columns)}; N = len(basket)
    rows = []
    for ant, con in zip(rules_df["antecedents"], rules_df["consequents"]):
        a = B[:, [idx[x] for x in ant]].all(axis=1); c = B[:, [idx[x] for x in con]].all(axis=1)
        n_A, n_C, n_AC = int(a.sum()), int(c.sum()), int((a & c).sum())
        conf = n_AC / n_A if n_A else np.nan
        lift = conf / (n_C / N) if n_A and n_C else np.nan
        rows.append((n_A, n_C, n_AC, N, conf, lift))
    return pd.DataFrame(rows, columns=["t_n_A", "t_n_C", "t_n_AC", "t_N", "test_conf", "test_lift"], index=rules_df.index)

def validate_split(train_raw, test_raw):
    edges, med = fit_bins(train_raw)
    _, r = mine_rules(make_basket(train_raw, edges, med))
    if r.empty:
        return r
    return pd.concat([r, evaluate_rules_on(make_basket(test_raw, edges, med), r)], axis=1)

def holdout_summary(res):
    ok = res.dropna(subset=["test_lift"])
    rho = spearmanr(ok["lift"], ok["test_lift"])[0] if len(ok) > 3 else np.nan
    return {"rules mined on train": len(res),
            "evaluable on test": len(ok),
            "% still lift > 1 on test": 100 * (ok["test_lift"] > 1).mean(),
            "% still conf >= min on test": 100 * (ok["test_conf"] >= MIN_CONFIDENCE).mean(),
            "median train lift": ok["lift"].median(),
            "median test lift": ok["test_lift"].median(),
            "train-test lift correlation": rho}

idx_tr, idx_te = train_test_split(df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE)
hold_real = validate_split(df.loc[idx_tr], df.loc[idx_te])
sh = shuffled_copy(df, RANDOM_STATE)
hold_shuf = validate_split(sh.loc[idx_tr], sh.loc[idx_te])

summary_hold = pd.DataFrame({"Real data": holdout_summary(hold_real), "Shuffled (no real associations)": holdout_summary(hold_shuf)}).round(3)
show_table(summary_hold)

top_cmp = hold_real.dropna(subset=["test_lift"]).head(10)[["rule", "support", "confidence", "lift", "test_conf", "test_lift", "t_n_A"]]
print("Top 10 training rules re-measured on the test set:")
show_table(top_cmp.round(3))

ok = hold_real.dropna(subset=["test_lift"])
plt.figure(figsize=(6.5, 5.5))
plt.scatter(ok["lift"], ok["test_lift"], alpha=0.5, edgecolors="k", linewidths=0.3)
lim = [0.5, max(ok["lift"].max(), ok["test_lift"].max()) * 1.05]
plt.plot(lim, lim, "r--", label="same lift on train and test"); plt.axhline(1, color="grey", linestyle=":")
plt.xlabel("Lift on training data"); plt.ylabel("Lift on unseen test data")
plt.title("Do rules keep their lift on new data?"); plt.legend(); plt.tight_layout(); plt.show()

def cross_validate_rules(frame):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    parts = []
    for fold, (tr, te) in enumerate(kf.split(frame), 1):
        res = validate_split(frame.iloc[tr], frame.iloc[te])
        if not res.empty:
            res = res.assign(fold=fold)
            parts.append(res[["rule", "fold", "lift", "t_n_A", "t_n_C", "t_n_AC", "t_N"]])
    allf = pd.concat(parts)
    g = allf.groupby("rule").agg(folds_mined=("fold", "nunique"), mean_train_lift=("lift", "mean"),
                                 nA=("t_n_A", "sum"), nC=("t_n_C", "sum"), nAC=("t_n_AC", "sum"), N=("t_N", "sum"))
    g["oof_conf"] = g["nAC"] / g["nA"].replace(0, np.nan)
    g["oof_lift"] = g["oof_conf"] / (g["nC"] / g["N"])
    return g.reset_index()

cv_real = cross_validate_rules(df)
cv_shuf = cross_validate_rules(shuffled_copy(df, RANDOM_STATE + 1))

def robust(g):
    return g[(g["folds_mined"] >= N_FOLDS - 1) & (g["oof_lift"] >= MIN_LIFT) & (g["oof_conf"] >= MIN_CONFIDENCE)]

STABLE_LABEL = f"stable rules (in >= {N_FOLDS - 1} folds, out-of-fold lift and confidence >= min)"
cv_summary = pd.DataFrame({
    "Real data": {"distinct rules mined in any fold": len(cv_real),
                  f"rules mined in >= {N_FOLDS - 1} folds": int((cv_real['folds_mined'] >= N_FOLDS - 1).sum()),
                  "% with out-of-fold lift > 1": 100 * (cv_real["oof_lift"] > 1).mean(),
                  STABLE_LABEL: len(robust(cv_real))},
    "Shuffled (no real associations)": {"distinct rules mined in any fold": len(cv_shuf),
                  f"rules mined in >= {N_FOLDS - 1} folds": int((cv_shuf['folds_mined'] >= N_FOLDS - 1).sum()),
                  "% with out-of-fold lift > 1": 100 * (cv_shuf["oof_lift"] > 1).mean(),
                  STABLE_LABEL: len(robust(cv_shuf))},
}).round(2)
show_table(cv_summary)

rob = robust(cv_real).sort_values("oof_lift", ascending=False)
print("Stable rules (mined in most folds and still strong on held-out folds).")
print("Note: training folds overlap by 80%, so stability is not independent evidence; compare with the shuffled column above.")
show_table(rob[["rule", "folds_mined", "mean_train_lift", "oof_conf", "oof_lift", "nAC"]].head(12).round(3) if len(rob) else "None")

def mine_counts(frame):
    e, m = fit_bins(frame)
    _, r = mine_rules(make_basket(frame, e, m))
    if r.empty:
        return {"strong": 0, "significant": 0, "max_lift": np.nan}
    return {"strong": int((r["lift"] >= MIN_LIFT).sum()), "significant": int((r["q_value"] < ALPHA).sum()),
            "max_lift": r["lift"].max()}

obs = mine_counts(df)
null = pd.DataFrame([mine_counts(shuffled_copy(df, 1000 + s)) for s in range(N_SHUFFLES)])

null_table = pd.DataFrame({
    "Real data": [obs["strong"], obs["significant"], obs["max_lift"]],
    f"Shuffled: mean (n={N_SHUFFLES})": [null["strong"].mean(), null["significant"].mean(), null["max_lift"].mean()],
    "Shuffled: 95th percentile": [null["strong"].quantile(.95), null["significant"].quantile(.95), null["max_lift"].quantile(.95)],
}, index=[f"strong rules (lift >= {MIN_LIFT})", f"significant rules (q < {ALPHA})", "maximum lift"]).round(2)
show_table(null_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist(null["strong"], bins=10, color="lightgray", edgecolor="k"); axes[0].axvline(obs["strong"], color="red", lw=2, label="real data")
axes[0].set_title("Strong rules found in shuffled data"); axes[0].set_xlabel("Number of strong rules"); axes[0].legend()
axes[1].hist(null["max_lift"], bins=10, color="lightgray", edgecolor="k"); axes[1].axvline(obs["max_lift"], color="red", lw=2, label="real data")
axes[1].set_title("Highest lift found in shuffled data"); axes[1].set_xlabel("Max lift"); axes[1].legend()
plt.tight_layout(); plt.show()

n_rows = len(df)
print(f"Support check: with {n_rows} records, min support {MIN_SUPPORT} = {int(np.ceil(MIN_SUPPORT * n_rows))} records. "
      f"Median support of the {int(rules['strong'].sum())} strong rules = {rules.loc[rules['strong'], 'support'].median():.3f} "
      f"({int(rules.loc[rules['strong'], 'n_AC'].median())} records), so each rule rests on very few records.")


CNN_RESULTS = {"5-fold CV accuracy": None, "5-fold CV ROC AUC": None,
               "Test accuracy": None, "Majority-class baseline": 0.50}

print("CNN (food delivery, supervised) results you entered:")
for k, v in CNN_RESULTS.items():
    print(f"  {k:<26}: {v if v is not None else 'not entered'}")
cnn_given = {k: v for k, v in CNN_RESULTS.items() if v is not None and k != "Majority-class baseline"}
if cnn_given and CNN_RESULTS["Test accuracy"] is not None:
    gap = CNN_RESULTS["Test accuracy"] - CNN_RESULTS["Majority-class baseline"]
    print(f"\nCNN test accuracy is {gap:+.2f} vs. the majority-class baseline.")

real_robust = len(robust(cv_real)); shuf_robust = len(robust(cv_shuf))
comparison = pd.DataFrame({
    "CNN (delivery prediction)": [
        "Supervised classification (Fast / Delayed)",
        "Map-style images built from locations and delivery details",
        "Accuracy, F1, ROC AUC, cross-validation, confusion matrix",
        "Needs hundreds to thousands of labeled examples",
        "Hard to interpret (black box)",
        "Can collapse to predicting one class and score ~50%"],
    "Apriori (pollution rules)": [
        "Unsupervised pattern mining (If-Then rules)",
        "Categorical baskets (Low / Medium / High items)",
        "Support, confidence, lift, significance, out-of-fold lift, shuffle control",
        "Works on small tables, but small data gives unstable rules",
        "Very interpretable (readable rules)",
        f"Finds many rules even in noise ({shuf_robust} 'stable' rules in shuffled data vs {real_robust} in real data)"],
}, index=["Task type", "Input representation", "How it is validated", "Data needed", "Interpretability", "Typical failure on noisy data"])
show_table(comparison)

n_sig = int((rules["q_value"] < ALPHA).sum())
corr_real = summary_hold.loc["train-test lift correlation", "Real data"]
corr_shuf = summary_hold.loc["train-test lift correlation", "Shuffled (no real associations)"]
sig_beats_null = obs["significant"] > null["significant"].quantile(.95)
carries_over = (not np.isnan(corr_real)) and corr_real > 0.10
beats_noise = bool(sig_beats_null and carries_over)

print("=" * 80)
print("SUMMARY (generated from this run's results)")
print("=" * 80)
print(f"1. Mined {len(freq_itemsets)} frequent itemsets and {len(rules)} rules; {int(rules['strong'].sum())} are 'strong' (lift >= {MIN_LIFT}); "
      f"{n_sig} are significant after FDR correction (shuffled data: {null['significant'].mean():.1f} on average).")
print(f"2. Do rules carry over to unseen data? Correlation between training lift and test lift: {corr_real:.2f} on real data "
      f"vs {corr_shuf:.2f} on shuffled data (near 0 = rules do not carry over).")
print(f"3. Shuffle control: real data has {obs['strong']} strong rules and max lift {obs['max_lift']:.2f}; "
      f"shuffled data typically gives {null['strong'].mean():.0f} and {null['max_lift'].mean():.2f}. "
      f"Stable cross-validation rules: {real_robust} real vs {shuf_robust} shuffled.")
print("   Many 'strong' rules also appear in pure noise, so lift alone is not evidence; significance and hold-out checks matter.")
if beats_noise:
    print("   -> The data contains association structure beyond chance. The significant, stable rules are the ones worth examining.")
else:
    print("   -> The rules are NOT clearly distinguishable from noise. Treat them as hypotheses, not findings.")
if len(pr_rules):
    best = pr_rules.iloc[0]
    print(f"4. Highest-lift pollution -> recovery rule: {best['rule']} (lift {best['lift']:.2f}, support {best['support']:.2f}, q = {best['q_value']:.2f}"
          + ("; significant)." if best['q_value'] < ALPHA else "; NOT significant)."))
print()
print("Recommendations (hypotheses to test, not proven effects):")
if beats_noise:
    sig_set = set(rules.loc[rules["q_value"] < ALPHA, "rule"])
    cand = rob[rob["rule"].isin(sig_set)] if len(rob) else rob
    pick = cand if len(cand) else rob
    if len(pick):
        print(f"  - Pollution control: examine the pattern in this significant, stable rule first: {pick.iloc[0]['rule']}.")
    print("  - Validate each such rule on new years of data before changing policy (association is not causation).")
else:
    print("  - Pollution control: do not base policy on these rules; first collect more country-year data (more records, more years).")
    print("  - Add variables that could really drive pollution and recovery (energy mix, recovery technology, regulation).")
print("  - Delivery predictions (from the CNN project): fix data quality, add real weather/time/GPS data and more deliveries;")
print("    use the CNN only if it clearly beats the majority-class baseline and Logistic Regression on held-out data.")
print("  - Use the same discipline for both: hold-out data, cross-validation and a chance baseline before trusting any pattern.")
